In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "SOLUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,200.62,200.62,200.19,200.43,8099.652,2025-09-01 00:00:59.999999+00:00,1.623070e+06,3822,2579.174,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,200.44,200.57,200.36,200.57,2420.952,2025-09-01 00:01:59.999999+00:00,4.853247e+05,1832,967.795,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.003141,0.001745,0.001396,NaN,NaN
2,2025-09-01 00:02:00+00:00,200.57,200.58,200.21,200.36,2998.765,2025-09-01 00:02:59.999999+00:00,6.007899e+05,2143,1127.508,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.002510,0.000001,-0.002511,NaN,NaN
3,2025-09-01 00:03:00+00:00,200.37,200.44,200.24,200.24,1907.570,2025-09-01 00:03:59.999999+00:00,3.821583e+05,1852,766.376,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.009351,-0.003167,-0.006184,NaN,NaN
4,2025-09-01 00:04:00+00:00,200.25,200.25,199.65,199.66,32397.094,2025-09-01 00:04:59.999999+00:00,6.479208e+06,6276,3251.055,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.035951,-0.012919,-0.023032,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-20 05:02:20,839] A new study created in memory with name: no-name-0d027561-8f5b-40f2-bb7d-2653b62b0fe6


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:12<?, ?it/s]

Best trial: 0. Best value: 0.0302056:   0%|          | 0/50 [00:12<?, ?it/s]

Best trial: 0. Best value: 0.0302056:   2%|▏         | 1/50 [00:12<10:31, 12.88s/it]

...

Best trial: 11. Best value: 0.0432369:  98%|█████████▊| 49/50 [29:56<00:27, 27.99s/it]

Best trial: 11. Best value: 0.0432369:  98%|█████████▊| 49/50 [29:56<00:27, 27.99s/it]

Best trial: 11. Best value: 0.0432369: 100%|██████████| 50/50 [29:56<00:00, 30.40s/it]

Best trial: 11. Best value: 0.0432369: 100%|██████████| 50/50 [29:56<00:00, 35.93s/it]

[I 2026-03-20 05:32:17,409] Trial 49 finished with value: 0.03713573897029304 and parameters: {'n_estimators': 500, 'max_depth': 12, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': 0.8, 'bootstrap': True}. Best is trial 11 with value: 0.04323686769679483.

[optuna] best trial
value: 0.043237
params:
  n_estimators: 600
  max_depth: 15
  min_samples_split: 11
  min_samples_leaf: 2
  max_features: 0.8
  bootstrap: True


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final rf...


[training] done in 46.04s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.351794
Test IC:       0.012946
Train Rank IC: 0.071435
Test Rank IC:  -0.019923
Train RMSE:    0.002350
Test RMSE:     0.002519


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
mom_60              0.212429
mom_30              0.096570
atr_norm            0.053614
dist_ma_30          0.052153
vol_15              0.044974
range_5             0.037430
mom_10              0.036176
range_15            0.034145
vol_5               0.034103
macd_hist           0.028607
dist_ma_15          0.025810
vol_30              0.022740
dist_ma_5           0.020323
mom_15              0.019094
mom_3               0.018428
mom_x_imb           0.018203
bar_range           0.017743
imbalance_15        0.017486
trend_strength      0.016261
vol_regime_ratio    0.016120
mom_5               0.015958
imbalance_5         0.014300
range_ratio         0.014097
dom_sin             0.013949
dist_ma_15_z        0.012792
hour_cos            0.011735
trend_x_imb         0.011455
trades_z            0.010781
imbalance           0.009555
vol_ratio_5_30      0.009391
volume_z            0.008779
hour_sin            0.006899
mr_x_vol            0.006440
num_trades_

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/SOLUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/SOLUSDT__h5_model.joblib
[saved] features -> models/rf/SOLUSDT__h5_feature_cols.json
[saved] feature importance -> models/rf/SOLUSDT__h5_feature_importance.csv
[saved] metadata -> models/rf/SOLUSDT__h5_meta.json
